# Lektion 11: Neo4j & Cypher – die echte Graph-Datenbank



| | |
|---|---|
| **Zeit** | ~60 Minuten |
| **Schwierigkeit** | Mittel |
| **Voraussetzungen** | Lektion 10 |
| **Du baust** | Deine ersten Cypher-Abfragen – gegen echtes Neo4j (Docker) oder den Offline-Fallback |

## 🎯 Nach dieser Lektion kannst du:

- erklären, was eine Graph-Datenbank gegenüber dem RAM-GraphStore bietet (Persistenz, Abfragesprache)

- Neo4j per Docker starten (oder bewusst den Offline-Fallback nutzen)

- Cypher-Grundmuster lesen und schreiben (`MATCH ... RETURN`, Kanten-Muster)

- Python mit Neo4j verbinden – mit **MERGE** (idempotent!) und **Parametern** (Injection-Schutz)

- die lesende Sicherheits-Whitelist verstehen

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [ ]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


## 🧠 Konzept: Vom RAM in die Datenbank

**WARUM:** Dein GraphStore lebt im Arbeitsspeicher – Programm zu, Daten weg. Und echte Datenbanken können Abfragen, die unser einfacher Store nicht kann.

**WAS:** **Neo4j** ist die bekannteste Graph-Datenbank: persistent, mehrnutzerfähig, mit eigener Abfragesprache **Cypher**.

**Metapher Cypher = ASCII-Art:** In SQL beschreibst du Verknüpfungslogik abstrakt. In Cypher **zeichnest du das gesuchte Muster wörtlich**:

```cypher
MATCH (a:Person)-[:WORKS_FOR]->(c:Company) RETURN a, c
```

Klammern sind Nodes, `[:WORKS_FOR]` die Kante, `->` die Richtung – wer Graphen denken kann, kann Cypher **lesen**.

## Setup: Zwei Wege (du entscheidest)

**Weg A – Docker (empfohlen, wenn verfügbar):**
```bash
docker run -d --name neo4j-academy -p 7474:7474 -p 7687:7687 \\
  -e NEO4J_AUTH=neo4j/lernakademie neo4j:5
```
Danach im Browser: http://localhost:7474 (Login: `neo4j` / `lernakademie`).

**Weg B – Offline-Fallback (immer möglich):** Der GraphStore versteht einen kleinen lesenden Cypher-Subset – dieselben Muster, keine Installation. **Das Notebook läuft in beiden Fällen.**

In [ ]:
from lernakademie.graph_store import baue_beispiel_graph

# Vier Standard-Muster (funktionieren in Neo4j UND im Fallback):
ABFRAGEN = [
    ("Alle Personen", "MATCH (n:Person) RETURN n"),
    ("Eine bestimmte Person", "MATCH (n:Person) WHERE n.name = 'Anna' RETURN n"),
    ("Wer arbeitet wo?", "MATCH (a:Person)-[r:WORKS_FOR]->(c:Company) RETURN a, c"),
    ("Alles von Anna (mit Kantentyp)", "MATCH (a {name: 'Anna'})-[r]->(b) RETURN a, type(r), b"),
]

store = baue_beispiel_graph()
for name, abfrage in ABFRAGEN:
    ergebnis = store.run_cypher(abfrage)
    print(f"{name}  [{len(ergebnis)} Treffer]")

### 🔎 Was sehen wir hier?
Vier Muster, die du in fast jeder Graph-Anfrage wiederfindest: Label-Filter (`:Person`), Werte-Bedingung (`WHERE`), Kanten-Muster (`-[r:TYP]->`) und der `type(r)`-Trick. **TRY:** Ändere `'Anna'` zu `'Ben'` und beobachte die Trefferzahl.

In [ ]:
# SICHERHEIT: Der Fallback ist eine WHITELIST – nur lesende MATCH...RETURN!
for verboten in ("CREATE (n:Person {name: 'Boesewicht'})",
                 "MATCH (n) DETACH DELETE n"):
    try:
        store.run_cypher(verboten)
        print("FEHLER IM DESIGN: ausgefuehrt!", verboten)
    except ValueError as fehler:
        print("Korrekt abgelehnt:", verboten[:35], "->", str(fehler)[:50], "...")

**WARUM die Ablehnung wichtig ist:** Später (Lektion 12) lässt du ein **LLM Cypher generieren**. Ein manipulierter Prompt könnte sonst `DELETE` erzeugen. Die Prüfung muss **deterministischer Code** sein – niemals ein LLM (Prompt 12 erklärt warum).

## Python ↔ Neo4j: die Brücke (mit Fallback)

Die Brücke probiert echtes Neo4j; ist es nicht erreichbar, läuft derselbe Code gegen den Fallback:

In [ ]:
from lernakademie.graph_store import GraphStore

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORT = "lernakademie"

def verbinde():
    """Echtes Neo4j – oder None (dann nutzt der Aufrufer den Fallback)."""
    try:
        from neo4j import GraphDatabase
        treiber = GraphDatabase.driver(
            NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORT),
            connection_timeout=3)
        treiber.verify_connectivity()
        return treiber
    except Exception as fehler:
        print(f"Neo4j nicht erreichbar -> Offline-Fallback. ({str(fehler)[:50]}...)")
        return None

treiber = verbinde()
verwende_fallback = treiber is None
if verwende_fallback:
    store = baue_beispiel_graph()
else:
    treiber.close()
print("Modus:", "Fallback (GraphStore)" if verwende_fallback else "Echtes Neo4j")

## ✏️ Deine Aufgabe: Parametrisierte Abfrage + MERGE

Du schreibst die produktionsreife Version der Brücke: Abfragen mit Parametern (Injection-sicher) und idempotentes Laden.

**Anforderungen:**

- Schreibe `frage(cypher, **parameter)`: im Fallback mit `store.run_cypher`, sonst per `session.run(cypher, **parameter)` – Ergebnis als Liste von Dicts (`.data()`).

- Schreibe `lade_firmen_graph()`, das die Firma per **MERGE** anlegt (mehrfaches Ausführen → KEINE Duplikate) – im Fallback: GraphStore mit `find_node`-Check (Entity Resolution!).

- Rufe `lade_firmen_graph()` **zweimal** auf und beweise: Die Anzahl der Nodes ist identisch.

**Schreibe deinen Code in die folgende Zelle:**

In [ ]:
def frage(cypher, **parameter):
    # YOUR CODE HERE (Fallback oder session.run mit Parametern)
    pass

def lade_firmen_graph():
    # YOUR CODE HERE: MERGE bzw. Entity Resolution
    pass

lade_firmen_graph()
vorher = len(store.nodes()) if verwende_fallback else None
lade_firmen_graph()
# YOUR CODE HERE: Beweis (Anzahl unveraendert?)

### ✅ Richtig, wenn:

- [ ] Die Abfrage nutzt Parameter (`$name`), niemals f-Strings.

- [ ] Doppeltes Laden erzeugt keine Duplikate.

- [ ] Der Code läuft im Fallback UND würde mit echtem Neo4j funktionieren (gleiche Struktur).

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [ ]:
# Im Fallback: Anzahl muss nach zweimaligem Laden stabil sein:
if verwende_fallback:
    assert len(store.nodes()) == len(baue_beispiel_graph().nodes()), \
        "Fallback: keine Duplikate erlaubt!"
ergebnis = frage("MATCH (n:Person) WHERE n.name = 'Anna' RETURN n", name="Anna")
assert ergebnis, "Anna sollte gefunden werden."
print("Alle Prüfungen bestanden! ✅")

<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Parameter in Cypher: `MATCH (n:Person {{name: $name}}) RETURN n` – übergeben als `frage(cypher, name='Anna')`.
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

MERGE = "finde ODER lege an" – das Neo4j-Äquivalent von Entity Resolution.
</details>

<details><summary>💡 Hinweis 3 (oeffnen, wenn du nicht weiterkommst)</summary>

Im Fallback übernimmt `find_node` diese Rolle: nur anlegen, wenn nicht gefunden.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. SQL-JOIN vs. Kante: Wie viele "Sprünge" bräuchte eine relationale DB für „Wer arbeitet an AI Assistant?" – und warum skaliert Graph-Traversal besser?

2. Was müsste passieren, bevor dein Assistent **schreibend** auf Neo4j darf? (→ Freigabe! Lektion 15)

## ➡️ Weiter

**Weiter mit Lektion 12:** `notebooks/06_knowledge_graph/12_llm_extraktion_graph.ipynb` – der Graph füllt sich **selbst**: ein LLM extrahiert Entitäten und Beziehungen aus Text.